# Noise-augmented BERT fine-tuning — CMPE 586
Paired arm of the completed clean baseline. Protocol: `docs/training-protocol.md`;
preparation results: `docs/augmentation-preparation.md`. Uses the same seed-586
initial tensors, records, word targets, optimization budget and clean validation
selection. Epoch-specific noisy views replace selected rows in place.

Preparation performs zero optimizer updates. Both execution switches default to
`False`; running all cells with these defaults cannot launch training.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] != (3, 12) or version("torch") != "2.11.0+cu128":
    raise RuntimeError("Select the observed Colab 2026.07 GPU runtime for the paired run.")
PROJECT = Path("/content/cmpe586-turkish-punctuation")
REPOSITORY = "https://github.com/kckagancan/cmpe586-turkish-punctuation.git"
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
else:
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=PROJECT, text=True).strip()
    if dirty:
        raise RuntimeError("Review checkout changes before updating.")
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT),
                "-c", str(PROJECT / "requirements/colab.constraints.txt")], check=True)
sys.path.insert(0, str(PROJECT / "src"))
print("Code commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip())

## Project storage
One mount-if-needed cell. Persistent reads/writes use the designated project
folder. The existing clean run and shared initialization are retained. No cleanup
or permanent Drive deletion is performed by this notebook.

In [ ]:
import os

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this notebook.")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive", timeout_ms=600_000)
STORAGE_ROOT = Path("/content/drive/MyDrive/cmpe586-turkish-punctuation")
if not STORAGE_ROOT.is_dir():
    raise RuntimeError("The existing project storage is not mounted.")
SEED = 586
CONFIG = PROJECT / "configs/training-augmented.json"
PAIRED_CLEAN = STORAGE_ROOT / "training-v1/train-clean-seed-586-v1"
INITIALIZATIONS = STORAGE_ROOT / "initializations/training-v1"
VIEWS = STORAGE_ROOT / f"inputs/training-augmentation-v1/seed-{SEED}"
CALIBRATION_OUTPUT = STORAGE_ROOT / f"training-v1/calibration-augmented-seed-{SEED}-v1"
TRAIN_OUTPUT = STORAGE_ROOT / f"training-v1/train-augmented-seed-{SEED}-v1"
print("GPU:", torch.cuda.get_device_name(0))
print("Paired clean run:", PAIRED_CLEAN)
print("Augmented output:", TRAIN_OUTPUT)

## Prepare and verify inputs — zero training updates
Reuse the persistent clean input cache and exact shared initialization. Prepare
three epoch views with separate training corruption seeds. This reads train and
validation only; the augmentation builder itself reads train only. Existing view
outputs are verified, not overwritten. Local and Colab Arrow file hashes may
differ; encoded-content hashes provide a serialization-independent comparison.

In [ ]:
subprocess.run([sys.executable, "scripts/prepare_training.py", "--storage-root", str(STORAGE_ROOT),
                "--seed", str(SEED), "--download-model"], cwd=PROJECT, check=True)
view_command = [sys.executable, "scripts/prepare_augmentation.py", "--config", str(CONFIG),
                "--seed", str(SEED), "--output-directory", str(VIEWS), "--download-tokenizer"]
if (VIEWS / "manifest.json").exists():
    view_command.append("--verify-only")
subprocess.run(view_command, cwd=PROJECT, check=True)
prepared = json.loads((VIEWS / "manifest.json").read_text())
EXPECTED_ENCODING_DIGESTS = {
    "1": "6d82e08a7ad9aa633ee6c47f99c6345eac61c7ecbfb5307b44355fd2a4df650f",
    "2": "60f3c22ea6d25369a40a1378da49a74dfcc88b8231d912b882e2dbe1094aaae9",
    "3": "c1227f3153e7ef483efc242a6d18556c1befcb2125063a032d68e56c42fd6368",
}
for epoch, item in prepared["epochs"].items():
    if item["audit"]["encoded_content_sha256"] != EXPECTED_ENCODING_DIGESTS[epoch]:
        raise RuntimeError(f"Epoch {epoch} differs from the audited local training view.")
    a = item["audit"]
    print(f"Epoch {epoch}: {a['selected_view_rows']:,} noisy-view rows; "
          f"{a['changes']['changed_word_percent']:.4f}% changed words; "
          f"max length {a['tokenization']['length_including_special_tokens']['max']}")
print("Prepared view identity:", prepared["identity"])
print("Training updates in preparation: 0")
BASE_COMMAND = [sys.executable, "scripts/train.py", "--config", str(CONFIG), "--arm", "augmented",
                "--seed", str(SEED), "--augmentation-directory", str(VIEWS),
                "--paired-clean-run", str(PAIRED_CLEAN),
                "--initialization-directory", str(INITIALIZATIONS), "--download-model"]

In [ ]:
def stream_command(command):
    import signal
    with subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="", flush=True)
            result = process.wait()
        except KeyboardInterrupt:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
            raise
        if result:
            raise subprocess.CalledProcessError(result, command)

## Checkpoint storage inspection — zero training updates
Four tiny writes exercise the two fixed slots and restricted loading on the mounted project folder.
This leaves only two small state files and an inspection report; it deletes nothing.
A passing inspection does not establish full optimizer-resume or cloud durability.

In [ ]:
CHECKPOINT_INSPECTION = STORAGE_ROOT / "preparation/checkpoint-layout-v1/augmented-seed-586-v1"
inspection_command = [sys.executable, "scripts/inspect_checkpoint_storage.py",
                      "--output-directory", str(CHECKPOINT_INSPECTION)]
if (CHECKPOINT_INSPECTION / "inspection.json").exists():
    inspection_command.append("--resume")
subprocess.run(inspection_command, cwd=PROJECT, check=True)
storage_inspection = json.loads((CHECKPOINT_INSPECTION / "inspection.json").read_text())
if storage_inspection["status"] != "passed" or storage_inspection["slot_state_files"] != 2:
    raise RuntimeError("Checkpoint storage inspection did not complete.")
print("Two fixed slots verified; optimizer updates: 0")

## GPU calibration — manual, disabled
A separate 30-update calibration starts from the shared initialization, includes
the longest prepared view and reports timing/memory. Calibration weights are not
used for formal training. This step and Drive slot behavior have not yet been
exercised for the augmented arm. Review calibration before enabling full training.

In [ ]:
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    stream_command([*BASE_COMMAND, "--mode", "calibrate", "--calibration-steps", "30",
                    "--output-directory", str(CALIBRATION_OUTPUT)])
else:
    print("Calibration disabled. No training updates started.")

## Full augmented training — manual, disabled
Three complete epochs, 10,968 updates. Select by clean validation macro-F1.
The runner rejects differences from the completed clean run's numerical recipe,
initialization, inputs and recorded numerical environment. Checkpoints reuse
`checkpoints/slot-01` and `slot-02`; the last verified slot stays intact while the
other is rewritten. Drive upload completion remains a separate concern.

For an interruption, retain all paths/code/settings and set `RESUME=True`.
A completed run is never overwritten.

In [ ]:
RUN_FULL_TRAINING = False
RESUME = False
if RUN_FULL_TRAINING:
    from turkish_punctuation.training_io import sha256
    if storage_inspection["training_io_sha256"] != sha256(PROJECT / "src/turkish_punctuation/training_io.py"):
        raise RuntimeError("Checkpoint code changed after the storage inspection.")
    calibration = json.loads((CALIBRATION_OUTPUT / "calibration.json").read_text())
    calibration_run = json.loads((CALIBRATION_OUTPUT / "run.json").read_text())
    specification = calibration_run["specification"]
    if (calibration["identity"] != calibration_run["identity"]
        or calibration["optimizer_updates"] != 30
        or not calibration["longest_prepared_batch_included"]
        or specification["arm"] != "augmented" or specification["seed"] != SEED
        or specification["inputs"]["augmentation"]["identity"] != prepared["identity"]):
        raise RuntimeError("A matching augmented calibration is required.")
    special_sources = {"training-config.json": CONFIG,
                       "tokenizer-config.json": PROJECT / "configs/tokenizer.json",
                       "noise-config.json": PROJECT / "configs/noise.json"}
    for name, expected in specification["source_hashes"].items():
        if sha256(special_sources.get(name, PROJECT / name)) != expected:
            raise RuntimeError(f"Source/config changed since calibration: {name}")
    command = [*BASE_COMMAND, "--mode", "train", "--output-directory", str(TRAIN_OUTPUT)]
    if RESUME:
        command.append("--resume")
    stream_command(command)
else:
    print("Full training disabled. Preparation only.")

In [ ]:
summary_path = TRAIN_OUTPUT / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("AUGMENTED TRAINING COMPLETE")
    print("Epochs:", len(summary["history"]), "Updates:", summary["global_step"])
    print("Best epoch:", summary["best_epoch"], "Clean validation macro-F1:", summary["best_macro_f1"])
    print("Selected model:", TRAIN_OUTPUT / summary["best_model"])
else:
    print("No completed augmented run yet.")

## Completed-run handoff
After completion and audit, create a models/results archive using
`scripts/archive_training.py`, including the shared initialization. Download it
manually, verify the local copy and model loading, then clean identified Drive
copies. Optimizer slots are disposable after the completed models/results are
safely preserved. This notebook does not delete them automatically.